# Curriculum 02 · Lab 3 — BGE vs E5: Head-to-Head Retrieval

**Goal:** Run two local bi-encoder embedding models head to head on the same
corpus and the same 60 real Q/A pairs, and measure **answer-containment
recall@1** — the fraction of questions whose gold answer string appears inside
the single most-similar passage. The embedding model is the block that decides
*which* passages a query can ever reach: retrieval quality is bounded by
embedding quality before any retriever, prompt, or LLM gets a vote.

```
Models     : BGE (BAAI/bge-base-en-v1.5, local) vs E5 (intfloat/multilingual-e5-base, local)
Data       : rag-mini-wikipedia (first 400 passages, first 60 real Q/A pairs)
Metric     : answer-containment recall@1 (case-insensitive substring)
```

**Protocol — the two models differ on purpose:** BGE
(`embeddings/bge.BGEEmbedding`) is *trained* for cosine similarity and the
wrapper emits unit-length vectors (`normalize_embeddings=True`); E5
(`embeddings/e5.E5Embedding`) is trained with instruction prefixes, so the
wrapper prepends `"query: "` to questions and `"passage: "` to passages
automatically — and, unlike BGE, it does **not** normalize its output. Both
cosine matrices are L2-normalized here with numpy so the dot products are
comparable and the mean/median similarities live on the same unit-scale. No
LLM, no judge, no API: pure retrieval measurement on real Q/A pairs.

## 0 · Setup — environment, imports & repo paths

**WHAT:** Silences the model-download progress bars, imports every dependency
(numpy, pandas), and puts the repo-root component library on `sys.path` so this
notebook reuses `embeddings/bge.py` and `embeddings/e5.py` exactly like the lab
script.

**WHY:** Everything embeds **locally** with BGE and E5 via `sentence-transformers`
— no API embeddings anywhere. The env var `HF_HUB_DISABLE_PROGRESS_BARS` must be
set **before any third-party import** (`huggingface_hub` reads the flag at import
time), so the "Loading weights" progress bar never buries the comparison tables.

**Paths:** the next cell resolves the **repo root** automatically — it works
whether the kernel launches from the repo root (like the lab script,
`python curriculum/02-embeddings/03-model-comparison.py`) or from the notebook's
own folder (the Jupyter default) — and `cd`s into it so every path stays
repo-relative.

**WHAT TO EXPECT:** no output — just a clean import. The models themselves are
loaded lazily later.

In [1]:
# Lab-specific dependencies (already in requirements.txt — the install is a
# no-op safety net for fresh environments):
#   sentence-transformers -> local BGE + E5 embeddings (embeddings/{bge,e5}.py)
#   pandas                -> reads the passages/test parquet corpus
#   numpy                 -> L2 normalization + cosine scoring matrices
%pip install sentence-transformers pandas numpy


[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: /home/magus/.pyenv/versions/3.12.7/envs/magus/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

# Silence the "Loading weights" progress bar (transformers honors
# HF_HUB_DISABLE_PROGRESS_BARS, read at huggingface_hub import time — set it
# before any third-party import).
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

import numpy as np
import pandas as pd

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the kernel's
# working directory — this works whether the kernel launches from the repo
# root (like the lab script) or from the notebook's own folder (Jupyter's
# default) — then cd into it so every repo-relative path behaves exactly
# like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

from embeddings.bge import BGEEmbedding  # noqa: E402
from embeddings.e5 import E5Embedding  # noqa: E402

## 1 · Configuration — the comparison's knobs

**WHAT:** The module-level constants that define the experiment: how many
passages to embed, how many questions to answer, how wide the printed passage
previews may be, which BGE model variant to load, and the two corpus files.

**WHY:** These are the knobs you tweak to re-run the comparison. `NUM_PASSAGES
= 400` and `NUM_QUESTIONS = 60` keep CPU embedding time reasonable (two models
× 400 passages + 60 queries); `PREVIEW` only trims the printed snippets in the
disagreement table — it does not change retrieval.

In [3]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the comparison
# --------------------------------------------------------------------------
NUM_PASSAGES = 400  # first N passages of the corpus (embedding budget)
NUM_QUESTIONS = 60  # first N test questions with non-empty answers
PREVIEW = 170  # max characters shown around the answer in the example table
MODEL_NAME = "BAAI/bge-base-en-v1.5"

PASSAGES_PARQUET = "Data/corpus/rag-mini-wikipedia/passages.parquet"
TEST_PARQUET = "Data/corpus/rag-mini-wikipedia/test.parquet"

## 2 · Load — deterministic subsets of the corpus

**WHAT:** `load_subsets()` reads the first `NUM_PASSAGES` rows of
`passages.parquet` and the first `NUM_QUESTIONS` rows of `test.parquet` whose
answer is a non-empty string, in file order.

**WHY:** Zero randomness keeps the run reproducible — the same passages and the
same Q/A pairs every time. Skipping empty answers matters: a blank answer can
never be *contained* in a passage, so it would silently count as a guaranteed
miss for both models.

In [4]:
# --------------------------------------------------------------------------
# 2. Load — deterministic subsets of the corpus
# --------------------------------------------------------------------------
def load_subsets() -> tuple[list[str], list[tuple[str, str]]]:
    """Return (passages, qa_pairs) with zero randomness.

    Passages: first ``NUM_PASSAGES`` rows of ``passages.parquet``. Questions:
    first ``NUM_QUESTIONS`` rows of ``test.parquet`` whose answer is a
    non-empty string, in file order.
    """
    passages_df = pd.read_parquet(PASSAGES_PARQUET)
    passages = [str(p) for p in passages_df["passage"].head(NUM_PASSAGES).tolist()]

    test_df = pd.read_parquet(TEST_PARQUET)
    qa_pairs: list[tuple[str, str]] = []
    for question, answer in zip(test_df["question"], test_df["answer"]):
        if isinstance(answer, str) and answer.strip():
            qa_pairs.append((str(question), answer.strip()))
        if len(qa_pairs) >= NUM_QUESTIONS:
            break
    return passages, qa_pairs

## 3 · Score — embed once per model, cosine via normalized dot products

**WHAT:** The scoring layer: `l2_normalize` makes every vector row unit-length,
`score_matrix` embeds all passages and all queries once per model and returns
the (60 × 400) cosine-similarity matrix as a single matrix product,
`contains_answer` is the case-insensitive substring hit test, and `ModelScore`
bundles the per-question top-1 outcomes (index, similarity, hit) for one model.
`escape` and `passage_window` only shape the printed snippets;
`disagreement_indices` finds the questions where the two models' top-1 hits
differ.

**WHY:** BGE's wrapper normalizes its output but E5's does not — a raw E5
vector is several times longer than a unit vector, so raw dot products would
not be comparable. Normalizing both matrices makes every dot product a cosine
similarity and puts the mean/median similarities on the same scale, which is
what makes section [3] meaningful.

In [5]:
# --------------------------------------------------------------------------
# 3. Score — embed once per model, cosine via normalized dot products
# --------------------------------------------------------------------------
def l2_normalize(matrix: np.ndarray) -> np.ndarray:
    """Row-wise L2 normalization to unit length."""
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.maximum(norms, 1e-12)


def score_matrix(embedder: object, passages: list[str],
                 questions: list[str]) -> np.ndarray:
    """Return the (n_questions, n_passages) cosine-similarity matrix.

    E5's wrapper does not normalize its output (unlike BGE's); both matrices
    are L2-normalized here so that (a) dot product equals cosine similarity
    and (b) the mean/median similarity reported for each model lives on the
    same unit-scale.
    """
    passage_vecs = np.asarray(embedder.embed_documents(passages), dtype=np.float32)
    query_vecs = np.asarray([embedder.embed_query(q) for q in questions],
                            dtype=np.float32)
    return l2_normalize(query_vecs) @ l2_normalize(passage_vecs).T


def contains_answer(passage: str, answer: str) -> bool:
    """Answer containment: case-insensitive substring on stripped strings."""
    return answer.lower() in passage.lower()


class ModelScore:
    """Per-question retrieval outcomes for one embedding model."""

    def __init__(self, name: str, cosine: np.ndarray,
                 passages: list[str], qa_pairs: list[tuple[str, str]]) -> None:
        self.name = name
        self.cosine = cosine
        self.top1_idx = cosine.argmax(axis=1)
        rows = np.arange(cosine.shape[0])
        self.top1_sim = cosine[rows, self.top1_idx]
        self.hits = [contains_answer(passages[i], answer)
                     for i, (_, answer) in zip(self.top1_idx, qa_pairs)]

    def recall_at_1(self) -> float:
        return float(np.mean(self.hits))

    def mean_sim(self) -> float:
        return float(np.mean(self.top1_sim))

    def median_sim(self) -> float:
        return float(np.median(self.top1_sim))


def escape(s: str) -> str:
    """Make newlines visible so passage previews stay on one line."""
    return s.replace("\n", "\\n")


def passage_window(passage: str, answer: str, width: int = PREVIEW) -> str:
    """Preview the passage, centered on the answer span when it is present."""
    idx = passage.lower().find(answer.lower())
    if idx < 0:
        return escape(passage[:width])
    start = max(0, idx - width // 3)
    end = min(len(passage), idx + len(answer) + width // 2)
    preview = escape(passage[start:end])
    prefix = "..." if start > 0 else ""
    suffix = "..." if end < len(passage) else ""
    return f"{prefix}{preview}{suffix}"


def disagreement_indices(bge: ModelScore, e5: ModelScore, limit: int) -> list[int]:
    """Question indices where the two models disagree on the top-1 hit."""
    return [i for i, (hb, he) in enumerate(zip(bge.hits, e5.hits))
            if hb != he][:limit]

## 4 · Run — load the subsets and print the setup

**WHAT:** The first step of the experiment: load the deterministic subsets,
print the banner, and print what we are comparing — model names, output dims,
and the wrapper differences that make this comparison non-trivial.

**WHY:** The setup printout makes the run self-describing. The dimension probe
(`embed_query` on the first question) is also what lazily loads both models, so
after this cell the comparison needs no further downloads.

In [6]:
# --------------------------------------------------------------------------
# 4. Print the artifact — runnable demo
# --------------------------------------------------------------------------
passages, qa_pairs = load_subsets()
questions = [q for q, _ in qa_pairs]

print("=" * 66)
print("Lab 03 — BGE vs E5: head-to-head retrieval comparison")
print("=" * 66)

# --- [1] Setup -------------------------------------------------------
print(f"\n[1] Setup")
models = [("BGE", BGEEmbedding(model_name=MODEL_NAME)), ("E5 ", E5Embedding())]
dims: dict[str, int] = {}
for label, embedder in models:
    probe = embedder.embed_query(questions[0])
    dims[label.strip()] = len(probe)
print(f"    passages : {len(passages)} (first {NUM_PASSAGES} of rag-mini-wikipedia)")
print(f"    questions: {len(qa_pairs)} (first {NUM_QUESTIONS} with non-empty answers)")
print(f"    BGE : BAAI/bge-base-en-v1.5            dim={dims['BGE']}  "
      f"(wrapper normalizes output)")
print(f"    E5  : intfloat/multilingual-e5-base   dim={dims['E5']}  "
      f"(wrapper auto-prefixes 'query:'/'passage:', no normalization)")
print("    both matrices L2-normalized here so cosine is a fair dot product")

Lab 03 — BGE vs E5: head-to-head retrieval comparison

[1] Setup


    passages : 400 (first 400 of rag-mini-wikipedia)
    questions: 60 (first 60 with non-empty answers)
    BGE : BAAI/bge-base-en-v1.5            dim=768  (wrapper normalizes output)
    E5  : intfloat/multilingual-e5-base   dim=768  (wrapper auto-prefixes 'query:'/'passage:', no normalization)
    both matrices L2-normalized here so cosine is a fair dot product


## 5 · Embed — both models over the corpus and the questions

**WHAT:** The heavy step: for each model, embed all 400 passages and all 60
questions, score the full cosine matrix, and wrap it in a `ModelScore`.

**WHY:** Embedding happens **once per model** — both models stay in memory and
every later section reuses these matrices, so the comparison never re-embeds.
This cell takes a few minutes on CPU the first time (model loads + 460 texts
per model); the per-model timing is printed right here.

In [7]:
# --- [2]-[4] Run each model -------------------------------------------
results: dict[str, ModelScore] = {}
for label, embedder in models:
    t0 = time.perf_counter()
    cosine = score_matrix(embedder, passages, questions)
    elapsed = time.perf_counter() - t0
    results[label.strip()] = ModelScore(label.strip(), cosine,
                                        passages, qa_pairs)
    print(f"    embedded {len(passages)} passages + {len(questions)} queries "
          f"with {label.strip()} in {elapsed:.1f}s")

    embedded 400 passages + 60 queries with BGE in 3.6s


    embedded 400 passages + 60 queries with E5 in 3.9s


## 6 · Recall@1 — gold answer found in the top-1 passage

**WHAT:** For each model, the fraction of the 60 questions whose gold answer
string appears (case-insensitively) inside the single most-similar passage.

**WHY:** This is the headline metric of the lab — pure retrieval measurement
with no LLM and no judge. The hit/miss columns make the ~0.25 scores legible:
only 15–16 of 60 questions are answered by the top-1 passage alone.

In [8]:
# --- [2] Recall@1 -----------------------------------------------------
bge, e5 = results["BGE"], results["E5"]
print("\n[2] Recall@1 — gold answer found in the top-1 passage")
print(f"    {'model':<10}{'recall@1':>10}{'hits':>7}{'misses':>9}")
for score in (bge, e5):
    hits = sum(score.hits)
    print(f"    {score.name:<10}{score.recall_at_1():>10.3f}"
          f"{hits:>7}{len(score.hits) - hits:>9}")


[2] Recall@1 — gold answer found in the top-1 passage
    model       recall@1   hits   misses
    BGE            0.250     15       45
    E5             0.267     16       44


## 7 · Similarity — cosine of the retrieved (top-1) passage

**WHAT:** Mean and median cosine similarity of the passage each model retrieved
as top-1, over the 60 questions.

**WHY:** With both matrices normalized to unit length, the similarities are on
the same scale — and yet they are *not* comparable across models: E5's top-1
similarities sit systematically higher. Absolute similarity does not transfer
between embedding spaces; only rankings do.

In [9]:
# --- [3] Similarity of the retrieved (top-1) passage -------------------
print("\n[3] Similarity of the retrieved passage (top-1 cosine)")
print(f"    {'model':<10}{'mean':>8}{'median':>9}")
for score in (bge, e5):
    print(f"    {score.name:<10}{score.mean_sim():>8.3f}"
          f"{score.median_sim():>9.3f}")


[3] Similarity of the retrieved passage (top-1 cosine)
    model         mean   median
    BGE          0.694    0.700
    E5           0.840    0.845


## 8 · Disagreements — retrievals where the models differ

**WHAT:** Up to three questions where the two models disagree on whether the
top-1 passage contains the gold answer, each shown side by side: the question,
the gold answer, and each model's verdict, similarity, and a passage preview
centered on the answer span when present.

**WHY:** The recall tables hide *which* questions each model wins. Reading the
disagreements shows the failure modes in action — e.g. E5's prefixed query
matching the passage that literally contains the answer, while BGE lands on a
topically similar but wrong passage.

In [10]:
# --- [4] Side-by-side disagreements ------------------------------------
print("\n[4] Example retrievals where the models disagree")
picked = disagreement_indices(bge, e5, limit=3)
if not picked:
    print("    No disagreement found in this subset — both models agree "
          "on every question.")
for i in picked:
    question, answer = qa_pairs[i]
    print(f"\n    Q: {escape(question[:110])}")
    print(f"       gold answer: {answer!r}")
    for score in (bge, e5):
        verdict = "HIT " if score.hits[i] else "MISS"
        snippet = passage_window(passages[score.top1_idx[i]], answer)
        print(f"       {score.name:<4}{verdict} sim={score.top1_sim[i]:.3f}  "
              f"{snippet[:PREVIEW + 8]}{'...' if len(snippet) > PREVIEW + 8 else ''}")


[4] Example retrievals where the models disagree

    Q: How old was Lincoln in 1816?
       gold answer: 'seven'
       BGE MISS sim=0.675  Young Abraham Lincoln
       E5  HIT  sim=0.859  Lincoln was just seven years old when, in 1816, the family was forced to make a new start in Perry County (...

    Q: Did the scientific community not reserve great attention to his theory ?
       gold answer: 'yes'
       BGE HIT  sim=0.677  ...t service to science. If you would even get them to say yes or no to your conclusions it would help to clear the future progress. I believe some...
       E5  MISS sim=0.804  Near the end of his career Faraday proposed that electromagnetic forces extended into the empty space around the conductor. This idea was rejected by his fellow scientist

    Q: When did he publish a collection?
       gold answer: '1733'
       BGE MISS sim=0.640  * Bence Jones, Henry (1870). The Life and Letters of Faraday in 2 vols, Longmans.
       E5  HIT  sim=0.789  At Nurembe

## 9 · Takeaway — what the tables teach

**WHAT:** Print the lesson of the comparison: the winner, the recall gap, and
the two caveats that keep the numbers honest.

**WHY:** `recall@1` only measures whether the top passage *contains* the gold
answer — a coarse proxy (many rag-mini answers are 'yes'/'no'). And because
absolute similarity never transfers across models, the takeaway is about
*rankings*, not scores: E5 needs its `'query:'` / `'passage:'` prefixes plus
L2 normalization to be compared fairly against BGE at all.

In [11]:
# --- [5] Takeaway ------------------------------------------------------
print("\n[5] Takeaway")
winner = "BGE" if bge.recall_at_1() > e5.recall_at_1() else \
    "E5" if e5.recall_at_1() > bge.recall_at_1() else "neither"
print(f"    On {len(qa_pairs)} real questions over {len(passages)} passages, "
      f"{winner} wins recall@1 "
      f"(BGE {bge.recall_at_1():.3f} vs E5 {e5.recall_at_1():.3f}).")
print("    Recall@1 only measures whether the top passage *contains* the")
print("    gold answer — a coarse proxy (many answers here are 'yes'/'no').")
print("    E5's top-1 similarities sit higher (median "
      f"{e5.median_sim():.3f} vs {bge.median_sim():.3f}) even with both")
print("    matrices unit-length: absolute similarity does NOT transfer")
print("    across models, only rankings do. And E5 needs its 'query:'/")
print("    'passage:' prefixes plus L2 normalization to be compared")
print("    fairly against BGE at all.")


[5] Takeaway
    On 60 real questions over 400 passages, E5 wins recall@1 (BGE 0.250 vs E5 0.267).
    Recall@1 only measures whether the top passage *contains* the
    gold answer — a coarse proxy (many answers here are 'yes'/'no').
    E5's top-1 similarities sit higher (median 0.845 vs 0.700) even with both
    matrices unit-length: absolute similarity does NOT transfer
    across models, only rankings do. And E5 needs its 'query:'/
    'passage:' prefixes plus L2 normalization to be compared
    fairly against BGE at all.
